Implement and train basic word embedding using a limited text corpus in Python to investigate semantic similarities and relationships among words.

Text Preprocessing

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Reshape
from tensorflow.keras.utils import plot_model

corpus = [
    "The quick brown fox jumps over the lazy dog.",
    "The dog barks loudly.",
    "The fox is cunning.",
    "A quick brown fox is a cunning animal."
]

processed_corpus = [sentence.lower().split() for sentence in corpus]

print("Original Corpus:")
for s in corpus:
    print(s)
print("\nProcessed Corpus (lowercase and tokenized):")
for s in processed_corpus:
    print(s)

Original Corpus:
The quick brown fox jumps over the lazy dog.
The dog barks loudly.
The fox is cunning.
A quick brown fox is a cunning animal.

Processed Corpus (lowercase and tokenized):
['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog.']
['the', 'dog', 'barks', 'loudly.']
['the', 'fox', 'is', 'cunning.']
['a', 'quick', 'brown', 'fox', 'is', 'a', 'cunning', 'animal.']


In [ ]:
tokenizer = Tokenizer(oov_token="<unk>")
tokenizer.fit_on_texts(processed_corpus)

word_sequences = tokenizer.texts_to_sequences(processed_corpus)

vocabulary_size = len(tokenizer.word_index) + 1

max_sequence_length = max(len(seq) for seq in word_sequences)

print(f"Vocabulary Size: {vocabulary_size}")
print(f"Max Sequence Length: {max_sequence_length}")
print("\nWord Index (a few examples):")
for word, index in list(tokenizer.word_index.items())[:10]:
    print(f"{word}: {index}")

print("\nExample Word Sequences:")
for i, seq in enumerate(word_sequences):
    print(f"Sentence {i+1}: {seq}")

Continous Bag of Words

In [16]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(oov_token="<unk>")
tokenizer.fit_on_texts(processed_corpus)
word_sequences = tokenizer.texts_to_sequences(processed_corpus)
vocabulary_size = len(tokenizer.word_index) + 1
max_sequence_length = max(len(seq) for seq in word_sequences)

padded_sequences = pad_sequences(word_sequences, maxlen=max_sequence_length, padding='pre')

def generate_cbow_data(sequences, window_size, vocabulary_size):
    X, y = [], []
    for seq in sequences:
        for i, target_word_idx in enumerate(seq):
            if target_word_idx == 0:
                continue
            context_words = []
            for j in range(max(0, i - window_size), i):
                if seq[j] != 0:
                    context_words.append(seq[j])
            for j in range(i + 1, min(len(seq), i + window_size + 1)):
                if seq[j] != 0:
                    context_words.append(seq[j])

            if context_words:
                X.append(context_words)
                y.append(target_word_idx)
    return np.array(X), np.array(y)

window_size = 2

inputs = []
labels = []

for sentence_tokens in word_sequences:
    if len(sentence_tokens) < 2:
        continue

    for i in range(len(sentence_tokens)):
        target_word = sentence_tokens[i]
        context_words = []
        for j in range(max(0, i - window_size), min(len(sentence_tokens), i + window_size + 1)):
            if i == j:
                continue
            context_words.append(sentence_tokens[j])

        if context_words:
            inputs.append(context_words)
            labels.append(target_word)

max_context_length = max(len(x) for x in inputs)

X_cbow = pad_sequences(inputs, maxlen=max_context_length, padding='post')
y_cbow = tf.keras.utils.to_categorical(labels, num_classes=vocabulary_size)

print(f"\nCBOW Input shape (X_cbow): {X_cbow.shape}")
print(f"CBOW Output shape (y_cbow): {y_cbow.shape}")
print("\nCBOW Input (first 5):\n", X_cbow[:5])
print("CBOW Label (first 5, one-hot encoded):\n", y_cbow[:5])



CBOW Input shape (X_cbow): (25, 4)
CBOW Output shape (y_cbow): (25, 18)

CBOW Input (first 5):
 [[4 5 0 0]
 [2 5 3 0]
 [2 4 3 8]
 [4 5 8 9]
 [5 3 9 2]]
CBOW Label (first 5, one-hot encoded):
 [[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]


TF-IDF (Term Frequency-Inverse Document Frequency)

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
corpus_for_tfidf = [" ".join(s) for s in processed_corpus]

words_from_tokenizer = [word for word in tokenizer.word_index.keys() if word != '<unk>']

words_from_tokenizer.sort()

tfidf_vocabulary = {word: i for i, word in enumerate(words_from_tokenizer)}

vectorizer = TfidfVectorizer(vocabulary=tfidf_vocabulary, token_pattern=r'(?u)\b\w+\b')

tfidf_matrix = vectorizer.fit_transform(corpus_for_tfidf)

tfidf_array = tfidf_matrix.toarray()

feature_names = vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(tfidf_array, columns=feature_names)

print("TF-IDF Matrix (first 5 rows and columns):")
display(tfidf_df.head())
print(f"\nShape of TF-IDF matrix: {tfidf_matrix.shape}")

TF-IDF Matrix (first 5 rows and columns):


,a,animal.,barks,brown,cunning,cunning.,dog,dog.,fox,is,jumps,lazy,loudly.,over,quick,the
0,0.00000,0.0,0.000000,0.300103,0.000000,0.0,0.300103,0.0,0.242960,0.000000,0.380643,0.380643,0.0,0.380643,0.300103,0.485919
1,0.00000,0.0,0.702035,0.000000,0.000000,0.0,0.553492,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.448100
2,0.00000,0.0,0.000000,0.000000,0.549578,0.0,0.000000,0.0,0.444931,0.549578,0.000000,0.000000,0.0,0.000000,0.000000,0.444931
3,0.76173,0.0,0.000000,0.300278,0.300278,0.0,0.000000,0.0,0.243101,0.300278,0.000000,0.000000,0.0,0.000000,0.300278,0.000000



Shape of TF-IDF matrix: (4, 16)


In [12]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

cosine_sim_matrix = cosine_similarity(tfidf_matrix)

sim_df = pd.DataFrame(
    cosine_sim_matrix,
    index=[f"Doc {i+1}" for i in range(len(corpus_for_tfidf))],
    columns=[f"Doc {i+1}" for i in range(len(corpus_for_tfidf))]
)

print("Cosine Similarity Matrix (TF-IDF based):")
display(sim_df)

print("\nInterpreting Similarities (e.g., between Document 1 and others):")
for i, similarity in enumerate(cosine_sim_matrix[0]):
    if i == 0:
        continue
    print(f"  Similarity between Document 1 and Document {i+1}: {similarity:.4f}")

print("\nHigher values indicate greater semantic similarity.")

Cosine Similarity Matrix (TF-IDF based):


,Doc 1,Doc 2,Doc 3,Doc 4
Doc 1,1.000000,0.383845,0.324301,0.239293
Doc 2,0.383845,1.000000,0.199373,0.000000
Doc 3,0.324301,0.199373,1.000000,0.438216
Doc 4,0.239293,0.000000,0.438216,1.000000



Interpreting Similarities (e.g., between Document 1 and others):
  Similarity between Document 1 and Document 2: 0.3838
  Similarity between Document 1 and Document 3: 0.3243
  Similarity between Document 1 and Document 4: 0.2393

Higher values indicate greater semantic similarity.
